<a href="https://colab.research.google.com/github/Angelin-ak/Micro_project1_Campus-City-Emergency-Supply-Distribution/blob/main/micro_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from google.colab import files
import pandas as pd
import io
import pulp
import folium

# --- STEP 1: BUDGET & FILE UPLOAD ---
try:
    annual_budget = float(input("💰 Enter the Annual Allocated Budget (e.g., 1500000): "))
except ValueError:
    print("⚠️ Invalid input. Defaulting budget to $1,500,000.")
    annual_budget = 1500000.0

print("\nPlease upload: 'facilities.csv', 'warehouses.csv', and 'transportation_costs.csv'")
uploaded = files.upload()

required_files = ['facilities.csv', 'warehouses.csv', 'transportation_costs.csv']
if not all(f in uploaded for f in required_files):
    print("❌ ERROR: Missing required files. Check filenames exactly.")
else:
    # --- STEP 2: LOAD DATA ---
    df_facilities = pd.read_csv(io.BytesIO(uploaded['facilities.csv']))
    df_warehouses = pd.read_csv(io.BytesIO(uploaded['warehouses.csv']))
    df_transport = pd.read_csv(io.BytesIO(uploaded['transportation_costs.csv']))

    # --- STEP 3: DYNAMIC DATA PREPARATION ---
    days = 365
    facilities = df_facilities['Facility ID'].tolist()
    warehouses = df_warehouses['Warehouse ID'].tolist()

    # Map Coordinates directly from CSV (Saintgits Campus focus)
    f_coords = {row['Facility ID']: (row['Lat'], row['Lon']) for _, row in df_facilities.iterrows()}
    w_coords = {row['Warehouse ID']: (row['Lat'], row['Lon']) for _, row in df_warehouses.iterrows()}

    demands = {row['Facility ID']: row['Daily Demand'] * days for _, row in df_facilities.iterrows()}
    caps = {row['Warehouse ID']: row['Daily Capacity'] * days for _, row in df_warehouses.iterrows()}
    fixed_costs = {row['Warehouse ID']: (row['Construction Cost'] / 10) + (row['Operational Cost/Day'] * days) for _, row in df_warehouses.iterrows()}

    # SAFETY CHECK: Map transportation costs and handle missing rows
    costs = {}
    for w in warehouses:
        costs[w] = {}
        for f in facilities:
            mask = (df_transport['Warehouse ID'] == w) & (df_transport['Facility ID'] == f)
            row = df_transport[mask]
            if not row.empty:
                costs[w][f] = row['Cost'].values[0]
            else:
                costs[w][f] = 999.0 # Penalty cost for missing data
                print(f"⚠️ Warning: Missing cost for {w} to {f}. Using penalty value.")

    # --- STEP 4: MATHEMATICAL OPTIMIZATION (PuLP) ---
    model = pulp.LpProblem("Saintgits_Campus_Optimization", pulp.LpMinimize)

    y = pulp.LpVariable.dicts("Open", warehouses, cat='Binary')
    x = pulp.LpVariable.dicts("Route", (warehouses, facilities), lowBound=0)

    # Objective Function
    model += pulp.lpSum([fixed_costs[i] * y[i] for i in warehouses]) + \
             pulp.lpSum([costs[i][j] * x[i][j] for i in warehouses for j in facilities])

    # Constraints
    for j in facilities:
        model += pulp.lpSum([x[i][j] for i in warehouses]) == demands[j]
    for i in warehouses:
        model += pulp.lpSum([x[i][j] for j in facilities]) <= caps[i] * y[i]

    model += pulp.lpSum([y[i] for i in warehouses]) == 2 # Rule: Open exactly 2

    model.solve(pulp.PULP_CBC_CMD(msg=0))

    # --- STEP 5: STRATEGIC REPORT OUTPUT ---
    print("\n" + "="*95)
    print("                SAINTGITS CAMPUS: STRATEGIC DISTRIBUTION REPORT")
    print("="*95)
    total_val = pulp.value(model.objective)

    print(f"TOTAL ANNUAL SYSTEM COST: ${total_val:,.2f}")
    print(f"ANNUAL ALLOCATED BUDGET:  ${annual_budget:,.2f}")

    if total_val <= annual_budget:
        print(f"✅ SUCCESS: Project is UNDER BUDGET by ${(annual_budget - total_val):,.2f}")
    else:
        print(f"🚨 ALERT: Project is OVER BUDGET by ${(total_val - annual_budget):,.2f}")

    print("\n💡 OPTIMIZED ROUTE LOGIC")
    print("-" * 95)
    print(f"{'Facility':<15} | {'Source Hub':<12} | {'Unit Cost':<10} | {'Status'}")
    print("-" * 95)

    for j in facilities:
        selected_wh_list = [i for i in warehouses if x[i][j].varValue > 0]
        if selected_wh_list:
            selected_wh = selected_wh_list[0]
            print(f"{j:<15} | {selected_wh:<12} | ${costs[selected_wh][j]:<9.2f} | Routed")
    print("-" * 95)

    # --- STEP 6: DYNAMIC GEOSPATIAL MAP ---
    if pulp.LpStatus[model.status] == 'Optimal':
        print("\n🌐 Rendering Campus Supply Map (Saintgits & Surroundings)...")

        # Centering Map on Saintgits College (Pathamuttom)
        m = folium.Map(location=[9.5215, 76.5410], zoom_start=12, tiles='CartoDB positron')

        #

        # Draw Warehouses
        for i in warehouses:
            color = 'red' if y[i].varValue == 1 else 'gray'
            folium.Marker(location=w_coords[i],
                          icon=folium.Icon(color=color, icon='university', prefix='fa'),
                          popup=f"Warehouse: {i}").add_to(m)

        # Draw Facilities and optimal green routes
        for j in facilities:
            folium.CircleMarker(location=f_coords[j], radius=6, color='blue', fill=True, popup=j).add_to(m)
            for i in warehouses:
                if x[i][j].varValue > 0:
                    folium.PolyLine(locations=[w_coords[i], f_coords[j]],
                                    color='green', weight=4, opacity=0.8).add_to(m)
        display(m)

💰 Enter the Annual Allocated Budget (e.g., 1500000): 1500000

Please upload: 'facilities.csv', 'warehouses.csv', and 'transportation_costs.csv'


Saving facilities.csv to facilities.csv
Saving transportation_costs.csv to transportation_costs.csv
Saving warehouses.csv to warehouses.csv

                SAINTGITS CAMPUS: STRATEGIC DISTRIBUTION REPORT
TOTAL ANNUAL SYSTEM COST: $834,465.00
ANNUAL ALLOCATED BUDGET:  $1,500,000.00
✅ SUCCESS: Project is UNDER BUDGET by $665,535.00

💡 OPTIMIZED ROUTE LOGIC
-----------------------------------------------------------------------------------------------
Facility        | Source Hub   | Unit Cost  | Status
-----------------------------------------------------------------------------------------------
MED_CENTER      | WH_CHINGA    | $1.80      | Routed
ENG_BLOCK       | WH_CHINGA    | $1.90      | Routed
MBA_BLOCK       | WH_CHINGA    | $1.70      | Routed
HOSTEL_A        | WH_KTYM      | $4.30      | Routed
HOSTEL_B        | WH_CHINGA    | $1.90      | Routed
LIBRARY         | WH_CHINGA    | $1.80      | Routed
RESEARCH_LAB    | WH_CHINGA    | $2.00      | Routed
AMENITY_CTR     | WH_CHING